# Path-Orphan — end-to-end Colab runner (verified)
Detect **conditional / orphan essentials** (essential, conservation<0.1) in *Ralstonia solanacearum* GMI1000 using dN/dS + AFDB/Foldseek + cofitness (+ESM), on top of conservation.

**Measured so far (sandbox, real data):**
- **Bar:** conservation R@P30 in the rogue zone = **0.000** (140 essentials / 1,694 genes). Gate is absolute: any P≥0.30 head beating the permutation null.
- **dN/dS: clean NEGATIVE** — rogue essentials ω med 0.158 vs non-essentials 0.160 (identical). Selection intensity ≠ conditional lethality. So the live question this notebook answers on Colab is whether **structure (Foldseek) / cofitness / ESM** add what dN/dS could not.

**Phase split — GPU on for ONE step only.** Every script is `--smoke`-verified, idempotent, Drive-cached. Run Phase A on CPU, switch to GPU for ESM only, switch back for Phase C.

## Setup (CPU) — clone, Drive, feba.db, deps

In [ ]:
import subprocess, os
from pathlib import Path
REPO = Path('/content/cell'); BRANCH='claude/vectorize-gex-propensity-NRqBW'
if not REPO.exists():
    subprocess.run(['git','clone','-b',BRANCH,'https://github.com/Nikku03/cell.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','origin',BRANCH],check=True)
os.chdir(REPO)
from google.colab import drive; drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/path_orphan'); DRIVE.mkdir(parents=True,exist_ok=True)
FEBA_SRC = Path('/content/drive/MyDrive/cell_count_dynamics/multiorg/fitness_browser/feba.db')
if FEBA_SRC.exists() and not Path('/content/feba.db').exists():
    subprocess.run(['cp',str(FEBA_SRC),'/content/feba.db'],check=True)
print('feba.db present:', Path('/content/feba.db').exists())
subprocess.run(['pip','-q','install','pandas','pyarrow','scikit-learn','xgboost'],check=True)
def cache(name): return (DRIVE/name).exists()
def stash(glob_): import shutil; [shutil.copy(p, DRIVE/p.name) for p in Path('outputs/orphan').glob(glob_)]

## Phase A (CPU) — runs end-to-end now

In [ ]:
# verify every step's logic first (fast)
for s in ['bridge','baseline','dnds','foldseek','cofit','model','esm']:
    print('==', s); subprocess.run(['python',f'scripts/orphan_{s}.py','--smoke'],check=True)

In [ ]:
# Step 0 bridge  +  Step 2 baseline(THE BAR)  +  Step 3 dN/dS  (all real, CPU)
!python scripts/orphan_bridge.py   --real
!python scripts/orphan_baseline.py --real
!python scripts/orphan_dnds.py     --real
stash('bridge_*'); stash('baseline_*'); stash('dnds_*'); stash('proteins_*'); stash('uniprot_request_*')

In [ ]:
# Step 5 cofit (needs feba.db). Set --fb_org to the Fitness Browser orgId for
# Ralstonia GMI1000 (inspect: SELECT DISTINCT orgId FROM Cofit).
!python scripts/orphan_cofit.py --real --feba /content/feba.db --fb_org Ralstonia || echo 'set --fb_org from feba.db'
stash('cofit_*')

### Step 4 — Foldseek vs AFDB (CPU, one-time DB download)
Install foldseek, map RefSeq→UniProt (from `uniprot_request_*.txt`), pull AFDB PDBs, search vs AFDB-cluster reps, then grade.

In [ ]:
# 1) foldseek
!wget -q https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz && tar xzf foldseek-linux-avx2.tar.gz
# 2) RefSeq protein_id -> UniProt (UniProt idmapping REST), input: outputs/orphan/uniprot_request_*.txt
# 3) AFDB pull: for each UniProt acc -> https://alphafold.ebi.ac.uk/files/AF-<acc>-F1-model_v4.pdb  (queries/)
# 4) foldseek databases afdb_clust + easy-search:
#    !foldseek databases Alphafold/UniProt50 afdb tmp        # (~25 GB, once -> stash on Drive)
#    !foldseek easy-search queries/ afdb outputs/orphan/foldseek_result.m8 tmp \
#        --format-output 'query,target,fident,evalue,bits,alntmscore'
# 5) grade (target_anno.tsv = AFDB target -> product; plddt.tsv = query -> mean pLDDT)
!python scripts/orphan_foldseek.py --real --m8 outputs/orphan/foldseek_result.m8 \
    --plddt outputs/orphan/plddt.tsv --target_anno outputs/orphan/target_anno.tsv || echo 'run foldseek steps 2-4 first'
stash('foldhit_*')

## ⚠️ SWITCH RUNTIME → A100/L4 — Phase B (ESM) only
Runtime → Change runtime type → GPU, re-run **Setup**, then run the cell below.

In [ ]:
!pip -q install torch transformers
!python scripts/orphan_esm.py --real   # ~20 min A100; reads proteins_*.faa -> esm_*.parquet
stash('esm_*')

## ⚠️ SWITCH RUNTIME → CPU — Phase C (model + null + GATE)

In [ ]:
# restore any Drive-cached features into outputs/orphan/ after a runtime switch
import shutil
for p in DRIVE.glob('*_beril_RalstoniaGMI1000.parquet'): shutil.copy(p, 'outputs/orphan/'+p.name)
# combined model over ALL available features; permutation null; GATE decision
!python scripts/orphan_model.py --real --null_perms 100
stash('model_*')
import json; print(json.dumps(json.load(open('outputs/orphan/model_beril_RalstoniaGMI1000.json')), indent=2))

## Read the verdict
`model_*.json` → `gate_pass`:
- **true** — structure/cofit/ESM detect rogue essentials conservation+dN/dS cannot → orphans are tractable for $0; scale to more orgs and write up.
- **false** — no $0 feature separates rogue essentials above the null. Combined with the dN/dS negative, that means conditional/orphan essentiality is **intrinsic to environment** and only interventional data (a screen) can decide → this is the clean, publishable negative that justifies the bacterial-DepMap argument.

Either outcome is a result. The experiment is designed so the cheap test settles the expensive question.